In [0]:
# === Init cell with config (RUN FIRST) ===

# Bronze layer:
# convert json to delta tables
# store all data (keep all batches, overlapping data and duplicates)
# add meatadata (batch_id, batch_timestamp)

# Blob storage access
storage_account = "medatastorage"
container = "forem-data"
storage_key = ""
spark.conf.set(
    f"fs.azure.account.key.{storage_account}.blob.core.windows.net",
    storage_key
)

# ADLS Gen2 storage with RBAC authentication (Unity Catalog managed identity)
storage_account_hierarchical = "foremstorageaccount"
bronze_adsl_path = f"abfss://adf-bronze@{storage_account_hierarchical}.dfs.core.windows.net/"

dbutils.fs.ls(bronze_adsl_path)


[]

In [0]:
%sql
-- SHOW TABLES IN metadata;
-- DESCRIBE TABLE metadata.processed_batches;
-- select * from metadata.processed_batches;
-- DELETE FROM metadata.processed_batches

-- INSERT INTO metadata.processed_batches
-- (pipeline_name, batch_id, batch_folder, processed_at)
-- VALUES
-- ('silver', '2026-backfill-1', 'wasbs://forem-data@medatastorage.blob.core.windows.net/2026-02-14/', current_timestamp()),
-- ('silver', '2026-backfill-2', 'wasbs://forem-data@medatastorage.blob.core.windows.net/2026-02-19/', current_timestamp());

pipeline_name,batch_id,batch_folder,processed_at
bronze,2026-backfill-1,wasbs://forem-data@medatastorage.blob.core.windows.net/2026-02-14/,2026-02-24T16:13:04.644241Z
bronze,2026-backfill-2,wasbs://forem-data@medatastorage.blob.core.windows.net/2026-02-19/,2026-02-24T16:13:04.644241Z
silver,2026-backfill-1,,2026-02-24T16:51:06.71031Z
silver,2026-backfill-2,,2026-02-24T16:51:06.71031Z


In [0]:
from pyspark.sql.functions import lit, current_timestamp

# List all top-level folders / files
paths = dbutils.fs.ls(f"wasbs://{container}@{storage_account}.blob.core.windows.net/")
date_folders = [p.path for p in paths if p.isDir()]
pipeline_name = "bronze"

# Processed data
processed_batches_folders = spark.sql(f"""
    SELECT batch_folder
    FROM metadata.processed_batches
    WHERE pipeline_name = '{pipeline_name}'
    """)
processed_batches = [row.batch_folder for row in processed_batches_folders.collect()]

current_batches = []

for folder in date_folders:
    if folder in processed_batches:
        print(f"Skip folder: {folder}")
        continue

    df_raw = spark.read.option("multiline", "true").json(folder)
    # Add metadata columns
    df_raw = (
        df_raw.withColumn("batch_id", lit("forem-api-folder"))
        .withColumn("source", lit("Forem API"))
        .withColumn("batch_ingestion_date", current_timestamp())
    )
    num_rows = df_raw.count()
    print(f"Number of rows in {folder}: {num_rows}")
    df_raw.write.format("delta").mode("append").save(bronze_adsl_path)
    current_batches.append(folder)

# Write metadata
print(f"Writing metadata for current batches: {current_batches}")
for current_batch in current_batches:
    batch_id = current_batch.rstrip("/").split("/")[-1]
    print('batch_id', batch_id)
    spark.sql(f"""
        MERGE INTO metadata.processed_batches t
        USING (
        SELECT
            'bronze' AS pipeline_name,
            '{batch_id}' AS batch_id,
            '{current_batch}' AS batch_folder,
            current_timestamp() AS processed_at
        ) s
        ON t.pipeline_name = s.pipeline_name
        AND t.batch_id = s.batch_id

        WHEN NOT MATCHED THEN
        INSERT (pipeline_name, batch_id, batch_folder, processed_at)
        VALUES (s.pipeline_name, s.batch_id, s.batch_folder, s.processed_at)
        """)



Skip folder: wasbs://forem-data@medatastorage.blob.core.windows.net/2026-02-14/
Skip folder: wasbs://forem-data@medatastorage.blob.core.windows.net/2026-02-19/
['wasbs://forem-data@medatastorage.blob.core.windows.net/2026-02-15/', 'wasbs://forem-data@medatastorage.blob.core.windows.net/2026-02-16/', 'wasbs://forem-data@medatastorage.blob.core.windows.net/2026-02-17/', 'wasbs://forem-data@medatastorage.blob.core.windows.net/2026-02-18/']
batch_id 2026-02-15
batch_id 2026-02-16
batch_id 2026-02-17
batch_id 2026-02-18
